## Setup

*You must run the cells in this section each time you connect to a new runtime. For example, when you return to the notebook after an idle timeout, when the runtime crashes, or when you restart or factory reset the runtime.*

Install requirements (*Note: ocdskingfishercolab installs google-colab, which expects specific versions of pandas and numpy*):

In [ ]:
! pip install --upgrade pip > pip.log
! pip install --upgrade ocdskingfishercolab ipywidgets psycopg2-binary >> pip.log

In [ ]:
# @title Import packages and load extensions { display-mode: "form" }

import gzip
import json
import os
import shutil
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from google.colab.data_table import DataTable
from google.colab.files import download
from ipywidgets import widgets
from ocdskingfishercolab import (
    authenticate_gspread,
    calculate_coverage,
    download_dataframe_as_csv,
    format_thousands,
    render_json,
    save_dataframe_to_sheet,
    save_dataframe_to_spreadsheet,
    set_dark_mode,
    set_light_mode,
)

# Load https://pypi.org/project/ipython-sql/
%load_ext sql
# Load https://colab.research.google.com/notebooks/data_table.ipynb
%load_ext google.colab.data_table

In [ ]:
# @title Configure the notebook environment { display-mode: "form" }

# Increase max columns so that Pandas DataFrames with many columns are rendered as data tables.
DataTable.max_columns = 50
# Remove the index from data tables for easier copy-pasting to Google Docs.
DataTable.include_index = False

# Return Pandas DataFrames instead of regular result sets.
%config SqlMagic.autopandas = True
# Don't print number of rows affected.
%config SqlMagic.feedback = False

# If you set Tools > Settings > Site > Theme to dark, uncomment this line.
# set_dark_mode()
# If you are creating plots to copy-paste into reports, uncomment this line.
# set_light_mode()

## Setup download data from the Data Registry

In [ ]:
# @title Data registry functions{ display-mode: "form" }
import requests

DATA_REGISTRY_BASE_URL = "https://data.open-contracting.org/en/"
PUBLICATIONS_URL = f"{DATA_REGISTRY_BASE_URL}publications.json"


def get_publications():
    publications = requests.get(PUBLICATIONS_URL, timeout=10).json()
    for publication in publications:
        publication["label"] = f"{publication['country']} - {publication['title']}"
    return publications


def get_publication_select_box():
    return widgets.Dropdown(
        options=sorted([entry["label"] for entry in get_publications()]),
        description="Publication:",
        disabled=False,
    )


def format_coverage(coverage):
    if not coverage:
        return pd.DataFrame(columns=["path"])
    fields = (
        pd.DataFrame.from_dict(coverage, orient="index", columns=["count"])
        .reset_index()
        .rename(columns={"index": "path"})
    )
    # Leaves only object members
    fields_table = fields[fields.path.str.contains("[a-z]$")].copy()
    fields_table["path"] = fields_table["path"].str.replace(r"[][]|^/", "", regex=True)
    return fields_table

## Select a publication from the [Data Registry](https://data.open-contracting.org/) and its field list

In [ ]:
# @title Select the publication to download { display-mode: "form" }

publication_select_box = get_publication_select_box()
publication_select_box

In [ ]:
# @title Extract the list of available fields { display-mode: "form" }

selected_publication = next(entry for entry in get_publications() if entry["label"] == publication_select_box.value)
fields_table = format_coverage(selected_publication.get("coverage", {}))

## Red flags analysis setup

Use this section to setup the functions needed to perform a usability analysis of the dataset, to identify if a publisher has the necessary fields to calculate 73 red flags indicators.

In [ ]:
# @title Red flags functions { display-mode: "form" }


def check_red_flags_indicators(result):
    # NEW Red Flags to OCDS mapping #Public
    spreadsheet_key = "1GACSPd64X5Tm-nu6LKttyEpaEp1CLsaCUGrEutljnFU"
    rows = authenticate_gspread().open_by_key(spreadsheet_key).get_worksheet(1).get_all_values()
    indicators = pd.DataFrame(rows).pipe(lambda df: df.rename(columns=df.iloc[0]).drop(df.index[0]))
    return result.merge(indicators.iloc[:, [0, 5, 6, 7]], on="R_id")

## Usability analysis

Generate a list of the fields published:

In [ ]:
fields_list = set(fields_table["path"])
indicators = load_indicators(prefix="R")
result = indicator_checks(fields_list, indicators)
result = result.rename(columns={"id": "R_id", "indicator": "red_flag"})

### Export results

#### Load use case indicators spreadsheet

In [ ]:
result_final = check_red_flags_indicators(result)

#### Table of results

In [ ]:
result_final

#### Results summary

In [ ]:
table = result_final.groupby("calculation").agg(total_red_flags=("R_id", "count")).reset_index()
table["%"] = round(table["total_red_flags"] / table["total_red_flags"].sum() * 100, 1)
table

#### Most common fields to indicators

In [ ]:
common_fields = most_common_fields_to_calculate_indicators(fields_list, indicators)
common_fields

#### Save the table to a spreadsheet

In [ ]:
spreadsheet_name = input("Enter the name of your spreadsheet:")
save_dataframe_to_sheet(spreadsheet_name, result_final, "red_flags_table")
save_dataframe_to_sheet(spreadsheet_name, common_fields, "common_fields_table")
save_dataframe_to_sheet(spreadsheet_name, fields_table, "fields_list")